In [289]:
import pandas as pd

In [290]:
from datetime import datetime

In [291]:
import random

In [292]:
random.seed(42)

load in time intervals csv 

In [293]:
time_intervals = pd.read_csv("./data/radarV1_MDD/csv_files/timeIntervals.csv")

In [294]:
time_intervals.head()

,timeInterval_ID,datetimeStart,datetimeEnd
0,11,2016-01-01 00:00:00,2016-01-01 23:59:59
1,12,2016-01-02 00:00:00,2016-01-02 23:59:59
2,13,2016-01-03 00:00:00,2016-01-03 23:59:59
3,14,2016-01-04 00:00:00,2016-01-04 23:59:59
4,15,2016-01-05 00:00:00,2016-01-05 23:59:59


In [295]:
time_intervals["date"] = pd.to_datetime(time_intervals["datetimeStart"]).dt.date

In [296]:
time_intervals['date'] = pd.to_datetime(time_intervals['date'], errors='coerce')

In [297]:
time_intervals["date"].dtype

dtype('<M8[ns]')

In [298]:
time_intervals.drop(columns = {"datetimeStart", "datetimeEnd"}, inplace = True)

remove phq8 duplicates

In [299]:
phq8_data = pd.read_csv("./data/radarV1_MDD/csv_files/phq8_data.csv")

In [300]:
phq8_data.shape

(13742, 48)

In [301]:
counts = (
    phq8_data.groupby(['participant_name', 'time_interval'])
      .size()
      .reset_index(name='count')
)

# Get the maximum count of non-unique time_interval per participant
max_counts = (
    counts.groupby('participant_name')['count']
          .max()
          .reset_index(name='max_nonunique_time_interval_count')
)
max_counts["max_nonunique_time_interval_count"].unique()

array([1, 2, 3])

In [302]:
cols = phq8_data.filter(regex=r'^value[0-7]$').columns

# Create the total_score column
phq8_data['total_score'] = phq8_data[cols].sum(axis=1)

In [303]:
phq8_data['score_range'] = (
    phq8_data.groupby(['participant_name', 'time_interval'])['total_score']
      .transform(lambda x: x.max() - x.min())
)

In [304]:
temp = phq8_data[["participant_name","time_interval", "total_score", "score_range"]]
temp = temp[temp["score_range"]> 8]
temp

,participant_name,time_interval,total_score,score_range
5417,3ca4e3d8-a922-4413-af5e-6cea14c720e1,1256,21,21
5418,3ca4e3d8-a922-4413-af5e-6cea14c720e1,1256,0,21
11303,078037ac-adec-44d4-be1b-93a4f1d4f765,1752,1,9
11304,078037ac-adec-44d4-be1b-93a4f1d4f765,1752,10,9
11317,078037ac-adec-44d4-be1b-93a4f1d4f765,1850,17,9
11318,078037ac-adec-44d4-be1b-93a4f1d4f765,1850,8,9
11319,078037ac-adec-44d4-be1b-93a4f1d4f765,1864,19,10
11320,078037ac-adec-44d4-be1b-93a4f1d4f765,1864,9,10
11321,078037ac-adec-44d4-be1b-93a4f1d4f765,1878,16,10
11322,078037ac-adec-44d4-be1b-93a4f1d4f765,1878,6,10


In [305]:
# when a participant has multiple scores in one day, keep the scores only if the range of scores for that day is less than max_range
max_range = 3
phq8_data = phq8_data[phq8_data['score_range'] <= max_range] 

In [306]:
phq8_data.shape # 32 deleted

(13710, 50)

In [307]:
# Randomly select 1 row per (participant_name, time_interval)
phq8_data = (
    phq8_data.groupby(['participant_name', 'time_interval'], group_keys=False)
      .apply(lambda x: x.sample(n=1))
      .reset_index(drop=True)
)

In [308]:
counts = (
    phq8_data.groupby(['participant_name', 'time_interval'])
      .size()
      .reset_index(name='count')
)

# Filter combinations that occur more than once
duplicates = counts[counts['count'] > 1]

print(duplicates)

Empty DataFrame
Columns: [participant_name, time_interval, count]
Index: []


In [309]:
phq8_data.shape

(13594, 50)

In [310]:
phq8_data.rename(columns={"time_interval": "timeInterval_ID"}, inplace = True)

In [311]:
# merge phq8 data with timeIntervals csv
phq8_data = phq8_data.merge(time_intervals, on="timeInterval_ID", how="left")

In [312]:
#compute the difference in dates within each ID group
#phq8_data = phq8_data.sort_values(['participant_name', 'date'])
##phq8_data['days_since_prev'] = phq8_data.groupby('participant_name')['date'].diff().dt.days

In [313]:
#temp = phq8_data[["participant_name", "date", "days_since_prev"]]

In [314]:
#temp[temp["participant_name"] == "007751c5-d7ad-4bec-a58f-abf32500e2ae"].head(10)

In [315]:
# sort
#phq8_data = phq8_data.sort_values(['participant_name', 'date'])

# Get each participant's first date
phq8_data['start_date'] = phq8_data.groupby('participant_name')['date'].transform('min')

# Compute period index (nearest 14-day interval per participant)
phq8_data['period_index'] = (
    ((phq8_data['date'] - phq8_data['start_date']).dt.days / 14)
    .round()
    .astype(int)
)

# Create reference date per participant
phq8_data['ref_date'] = phq8_data['start_date'] + pd.to_timedelta(phq8_data['period_index'] * 14, unit='D')

# Difference in days
phq8_data['diff_days'] = (phq8_data['date'] - phq8_data['ref_date']).dt.days

# Filter within ±2 days
max_difference_from_reference = 2
phq8_data = phq8_data[phq8_data['diff_days'].abs() <= max_difference_from_reference]

In [316]:
phq8_data.shape

(13211, 55)

In [317]:
temp = phq8_data[["participant_name", "date", "start_date","period_index", "ref_date", "diff_days"]].sort_values(['participant_name', 'date'])
temp[temp["participant_name"] == "007751c5-d7ad-4bec-a58f-abf32500e2ae"].head(10)

,participant_name,date,start_date,period_index,ref_date,diff_days
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-01-07,2019-01-07,0,2019-01-07,0
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-01-21,2019-01-07,1,2019-01-21,0
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-02-04,2019-01-07,2,2019-02-04,0
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-03-04,2019-01-07,4,2019-03-04,0
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-04-01,2019-01-07,6,2019-04-01,0
5,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-05-01,2019-01-07,8,2019-04-29,2
6,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-05-27,2019-01-07,10,2019-05-27,0
7,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-06-12,2019-01-07,11,2019-06-10,2
9,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-09-17,2019-01-07,18,2019-09-16,1
10,007751c5-d7ad-4bec-a58f-abf32500e2ae,2019-10-16,2019-01-07,20,2019-10-14,2


In [318]:
duplicates = phq8_data[phq8_data.duplicated(['participant_name', 'period_index'], keep=False)]
duplicates[["participant_name", "period_index", "date", "ref_date", "total_score", "diff_days"]].head(5)

,participant_name,period_index,date,ref_date,total_score,diff_days
158,0157930f-10f7-4d5f-9575-a65efbd1d505,23,2020-06-11,2020-06-10,2,1
159,0157930f-10f7-4d5f-9575-a65efbd1d505,23,2020-06-12,2020-06-10,2,2
534,08654850-1e57-450d-9341-ae8bca241c08,26,2020-04-10,2020-04-10,9,0
535,08654850-1e57-450d-9341-ae8bca241c08,26,2020-04-12,2020-04-10,8,2
946,146f6d46-d90b-43fb-b439-93c456104fc3,41,2020-04-22,2020-04-22,14,0


In [319]:
# when multiple phq8 scores align to the same period, keep the score that took place closest to the refernce date
phq8_data = (
    phq8_data.sort_values('diff_days', key=abs)
      .drop_duplicates(['participant_name', 'period_index'])
)

In [320]:
duplicates = phq8_data[phq8_data.duplicated(['participant_name', 'period_index'], keep=False)]
duplicates

,participant_name,file_name,file_size,file_modified_on,project_id,user_id,source_id,time,time_completed_utc,time_completed_local,...,end_time7,created_at,updated_at,total_score,score_range,date,start_date,period_index,ref_date,diff_days


In [321]:
phq8_data.shape

(13122, 55)

In [322]:
13122/13742

0.9548828409256295

In [323]:
    
relevant_columns = [
        'participant_name',   # Unique identifier for each participant
        'timeInterval_ID',    # ID linking the record to a specific time interval
        'date',               # Date of the PHQ-8 assessment 
        'total_score',        # Sum of PHQ-8 item scores 
        'start_date',         # First recorded assessment date for the participant (baseline)
        'period_index',       # Index of the nearest 14-day window since start_date
        'ref_date',           # Reference date for the 14-day window (start_date + 14 * period_index)
        'diff_days'           # Difference (in days) between actual date and reference date
    ]
    
phq8_data = phq8_data[relevant_columns]

In [324]:
phq8_data.to_csv("./data_cleaned/phq8_data_notebook.csv", index = False)

In [191]:
phq8_data.columns

Index(['participant_name', 'file_name', 'file_size', 'file_modified_on',
       'project_id', 'user_id', 'source_id', 'time', 'time_completed_utc',
       'time_completed_local', 'timeInterval_ID', 'time_notification', 'name',
       'version', 'question_id0', 'value0', 'start_time0', 'end_time0',
       'question_id1', 'value1', 'start_time1', 'end_time1', 'question_id2',
       'value2', 'start_time2', 'end_time2', 'question_id3', 'value3',
       'start_time3', 'end_time3', 'question_id4', 'value4', 'start_time4',
       'end_time4', 'question_id5', 'value5', 'start_time5', 'end_time5',
       'question_id6', 'value6', 'start_time6', 'end_time6', 'question_id7',
       'value7', 'start_time7', 'end_time7', 'created_at', 'updated_at',
       'total_score', 'score_range', 'date', 'days_since_prev', 'start_date',
       'period_index', 'ref_date', 'diff_days'],
      dtype='object')

In [192]:
assert phq8_data['total_score'].between(0, 24).all(), "Invalid PHQ8 scores"

merge timeintervals.csv with sleep data

In [106]:
sleep_data = pd.read_csv("./data/radarV1_MDD/csv_files/dailyFeatures_fitbit_sleep.csv")

In [111]:
sleep_data.head()

,participant_name,file_name,file_size,file_modified_on,project_id,user_id,time_interval,time,name,version,...,NREM_pct,sleep_onset,sleep_offset,rem_latency,sleep_efficiency,awakenings,insomnia,hypersomnia,created_at,updated_at
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,exported_sleep_daily_features/RADAR-MDD-CIBER-...,321,1634133433,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1208,300,sleep_features,1.0,...,78.174037,3.583333,9.350000,3.025000,92.296719,1,0,0,2021-10-13 18:34:43,2021-10-13 18:34:43
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,exported_sleep_daily_features/RADAR-MDD-CIBER-...,334,1634133436,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1209,300,sleep_features,1.0,...,72.131148,3.566667,8.633333,4.175000,89.344262,3,0,0,2021-10-13 18:34:43,2021-10-13 18:34:43
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,exported_sleep_daily_features/RADAR-MDD-CIBER-...,333,1634133434,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1210,200,sleep_features,1.0,...,67.502987,2.800000,9.616667,2.191667,93.548387,1,0,0,2021-10-13 18:34:43,2021-10-13 18:34:43
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,exported_sleep_daily_features/RADAR-MDD-CIBER-...,320,1634133435,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1211,200,sleep_features,1.0,...,70.312500,2.783333,9.033333,2.466667,91.276042,2,0,0,2021-10-13 18:34:43,2021-10-13 18:34:43
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,exported_sleep_daily_features/RADAR-MDD-CIBER-...,335,1634133437,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1211,2300,sleep_features,1.0,...,79.487179,23.816667,4.050000,1.883333,90.532544,2,0,0,2021-10-13 18:34:43,2021-10-13 18:34:43


In [112]:
sleep_data.shape

(183207, 26)

In [113]:
sleep_data.rename(columns={"time_interval": "timeInterval_ID"}, inplace = True)

In [114]:
sleep_data = sleep_data.merge(time_intervals, on="timeInterval_ID", how="left")

In [97]:
sleep_data.columns

Index(['participant_name', 'file_name', 'file_size', 'file_modified_on',
       'project_id', 'user_id', 'timeInterval_ID', 'time', 'name', 'version',
       'total_sleep_time', 'time_in_bed', 'light_pct', 'deep_pct', 'REM_pct',
       'awake_pct', 'NREM_pct', 'sleep_onset', 'sleep_offset', 'rem_latency',
       'sleep_efficiency', 'awakenings', 'insomnia', 'hypersomnia',
       'created_at', 'updated_at', 'datetimeStart', 'datetimeEnd', 'date'],
      dtype='object')

In [115]:
sleep_data = sleep_data[["participant_name", "timeInterval_ID", "date", "total_sleep_time", 
                         "time_in_bed", 'light_pct', 'deep_pct', 'REM_pct',
                         'awake_pct', 'NREM_pct', 'sleep_onset', 'sleep_offset', 'rem_latency',
                         'sleep_efficiency', 'awakenings', 'insomnia', 'hypersomnia']]

In [117]:
duplicates = sleep_data[sleep_data.duplicated(['participant_name', 'timeInterval_ID'], keep=False)]
duplicates.head(10)

,participant_name,timeInterval_ID,date,total_sleep_time,time_in_bed,light_pct,deep_pct,REM_pct,awake_pct,NREM_pct,sleep_onset,sleep_offset,rem_latency,sleep_efficiency,awakenings,insomnia,hypersomnia
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,1211,2019-04-15,21030,23040,54.296875,16.015625,20.963542,8.723958,70.312500,2.783333,9.033333,2.466667,91.276042,2,0,0
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,1211,2019-04-15,13770,15210,51.479290,28.007890,11.045365,9.467456,79.487179,23.816667,4.050000,1.883333,90.532544,2,0,0
17,007751c5-d7ad-4bec-a58f-abf32500e2ae,1245,2019-05-19,18150,19890,49.773756,20.211161,21.266968,8.748115,69.984917,4.200000,9.583333,1.925000,91.251885,2,0,0
18,007751c5-d7ad-4bec-a58f-abf32500e2ae,1245,2019-05-19,14520,15870,64.650284,13.232514,13.610586,8.506616,77.882798,19.016667,23.416667,3.400000,91.493384,1,0,0
26,007751c5-d7ad-4bec-a58f-abf32500e2ae,1288,2019-07-01,12750,14790,39.148073,26.774848,20.283976,13.793103,65.922921,5.600000,9.566667,1.450000,86.206897,2,0,0
27,007751c5-d7ad-4bec-a58f-abf32500e2ae,1288,2019-07-01,9720,10890,48.209366,24.793388,16.253444,10.743802,73.002755,17.266667,20.283333,1.841667,89.256198,2,0,0
39,007751c5-d7ad-4bec-a58f-abf32500e2ae,1414,2019-11-04,25080,26160,51.261468,19.724771,24.885321,4.128440,70.986239,1.466667,8.733333,3.166667,95.871560,2,0,0
40,007751c5-d7ad-4bec-a58f-abf32500e2ae,1414,2019-11-04,22410,25470,64.546525,10.482921,12.956419,12.014134,75.029446,12.566667,19.650000,0.875000,87.985866,1,0,0
41,007751c5-d7ad-4bec-a58f-abf32500e2ae,1414,2019-11-04,35910,40740,57.216495,8.394698,22.533137,11.855670,65.611193,21.766667,9.083333,1.383333,88.144330,4,0,0
55,007751c5-d7ad-4bec-a58f-abf32500e2ae,1459,2019-12-19,22560,24810,69.165659,12.575574,9.189843,9.068924,81.741233,2.100000,8.900000,5.166667,90.931076,3,0,0


In [118]:
sleep_data = (
    sleep_data.groupby(['participant_name', 'timeInterval_ID'], group_keys=False)
      .apply(lambda x: x.sample(n=1))
      .reset_index(drop=True)
)

In [119]:
duplicates = sleep_data[sleep_data.duplicated(['participant_name', 'timeInterval_ID'], keep=False)]
duplicates

,participant_name,timeInterval_ID,date,total_sleep_time,time_in_bed,light_pct,deep_pct,REM_pct,awake_pct,NREM_pct,sleep_onset,sleep_offset,rem_latency,sleep_efficiency,awakenings,insomnia,hypersomnia


aggregate sleep data over 2 weeks

merge demographic data with sleep data and phq8 data over 2 week intervals

In [3]:
total_dataset = pd.read_stata("./data/totaldataset.dta")

/var/folders/bt/f2dk62zd4lv80vqdphjgmw840000gn/T/ipykernel_4861/4226013150.py:1: UserWarning: Encountered %tC format. Leaving in Stata Internal Format.
  total_dataset = pd.read_stata("./data/totaldataset.dta")


In [6]:
for column in total_dataset.columns:
    print(column)

record_id
redcap_event_name
redcap_repeat_instrument
redcap_repeat_instance
subject_id
human_readable_id
radar_integration_complete
dob
age
gender
gender_at_birth
marital_status
living_together
children
no_of_children
children_live_with_you
youngest_child_age
ethnicity_1
white_other
mixed_other
asian_other
black_other
ethnic_other
country
eduction_finish_age
drop_school_1
qualifications
degree_1
benefits_type___0
benefits_type___1
benefits_type___2
benefits_type___3
benefits_type___4
benefits_type___5
benefits_type___6
benefits_type___7
benefits_type___8
benefits_type___9
benefits_type___10
benefits_type_other
benefits
income_gbp
income_contribution
accommodation
accommodation_satisfaction
utilities___0
utilities___1
utilities___2
utilities___3
utilities___4
utilities___5
utilities___6
utilities___7
utilities___8
utilities___9
utilities___10
socio_demographics_complete
social_environment_timestamp
contacts_weekly___0
contacts_weekly___1
contacts_weekly___2
contacts_weekly___3
contacts_